# Synthetic Heart Condition Classification

Version 2: five overlapping fictional labels. This notebook uses the active generated dataset and saved pipeline. It is not clinical diagnosis or medical risk prediction. Generator coefficients and prevalence are invented. The original notebook is preserved under `archive/binary_v1/notebooks/`.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd() if (Path.cwd() / 'health_data.py').exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from health_data import read_dataset, FEATURE_COLUMNS, TARGET_COLUMNS, example_inputs
from predict import load_artifacts, predict_patient
frame = read_dataset()
print(frame.shape)
frame.head()

## Provenance and labels

Run `python generate_data.py` and `python train_model.py` from the root to reproduce. This notebook does not overwrite datasets or retrain. Targets are sampled after features and can overlap.

In [ ]:
card = json.loads((ROOT / 'data/synthetic_heart_conditions.json').read_text())
print(card['source'])
print(card['label_process'])
print(card['limitations'])
frame[TARGET_COLUMNS].sum().rename('Positive synthetic labels')

In [ ]:
frame[TARGET_COLUMNS].sum().plot.bar(title='Fictional label counts, not clinical prevalence', rot=20)
plt.tight_layout()
plt.show()
print(frame[TARGET_COLUMNS].sum(axis=1).value_counts().sort_index())

## Integrity and preprocessing

A saved pipeline fits imputation, one-hot encoding and optional scaling only on training data. Input features exclude all five targets. The 64/16/20 random split is disjoint and seeded, with both classes checked for each target.

In [ ]:
assert set(FEATURE_COLUMNS).isdisjoint(TARGET_COLUMNS)
assert not frame[FEATURE_COLUMNS].duplicated().any()
assert (frame.Systolic_BP > frame.Diastolic_BP).all()
assert not frame.isna().any().any()
audit = json.loads((ROOT / 'results/data_audit.json').read_text())
print(audit['label_count_distribution'])

## Compare fitted models

Validation macro F1 selects the model; macro average precision breaks ties. The test set does not choose the winner. All thresholds are fixed at 0.5. Subset accuracy requires all five labels to match; it differs from per-label accuracy.

In [ ]:
scores = pd.read_csv(ROOT / 'results/model_comparison.csv')
print(scores[scores['split'].isin(['validation', 'test', 'test_missing_clinical_fields'])].to_string(index=False))
model, metadata = load_artifacts()
print('Selected:', metadata['model_name'])
print(metadata['selection_rule'])

In [ ]:
reports = json.loads((ROOT / 'results/classification_reports.json').read_text())
pd.DataFrame(reports[metadata['model_name']]['test']).T

## Fictional predictions

Example A demonstrates zero positive synthetic labels, and B demonstrates overlapping labels. These are software examples, not patient diagnoses. Five independent scores do not sum to 100%.

In [ ]:
for profile in ['A', 'B']:
    result = predict_patient(example_inputs(profile), model=model, metadata=metadata, advisory_units_confirmed=True)
    print(profile, result['message'])
    print(pd.DataFrame([{'condition': c['condition'], 'score': c['estimated_probability'], 'label': c['predicted_label']} for c in result['conditions']]).to_string(index=False))

## Missing clinical inputs and limits

Hiding four clinical test inputs lowers macro F1 to about 0.222 in the synthetic stress test. Imputation is not clinical testing. No score, positive label, negative label, or lifestyle topic establishes a diagnosis. Real-world condition identification requires appropriate clinical data and independent evaluation. See the current college report for the full methods and sources.